# Exercises XP Ninja: Guided Student Notebook

This guided notebook follows the **exercises on the platform**. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points are included only when a concept is important for intuition or transfer to other AI topics.


## Reference from the exercises

**What you will learn**  
- Implement state-of-the-art techniques to solve complex machine learning problems.
- Understand and apply advanced optimization methods for deep learning.
- Build and fine tune large scale models for real world applications.
- Explore cutting edge research areas like generative models and reinforcement learning.
- Develop strategies to handle imbalanced datasets and improve model robustness.

**What you will create**  
A deep learning model optimized using learning rate scheduling and gradient clipping. A generative model to create synthetic data. A reinforcement learning agent for a specific task. A robust model trained on an imbalanced dataset using SMOTE or focal loss. A comparison of performance before and after advanced strategies.


## Exercise 1: Advanced Optimization Techniques for Deep Learning

**As stated in the exercises**  
Objective. Improve model training stability and convergence using advanced optimization techniques.  
Instructions. Choose a deep learning model such as a CNN or RNN and a dataset such as CIFAR 10 or IMDB. Implement learning rate scheduling such as cosine annealing or step decay during training. Apply gradient clipping to prevent exploding gradients. Compare training stability and convergence with and without these techniques. Write a short analysis of how these techniques improve training.

**Guidance**  
Use CIFAR 10 with a small CNN for quick runs. Compare baseline vs scheduled learning rate and gradient clipping. Track training and validation curves.


**Learning point**  
Learning rate schedules reduce the step size as training progresses which helps convergence near minima. Cosine schedules can give sharper early progress. Gradient clipping caps update magnitude to prevent unstable jumps, especially in RNNs or deep CNNs.


In [ ]:
# PREFILLED: just execute
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)

# CIFAR-10 data
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
x_tr = x_tr.astype("float32")/255.0
x_te = x_te.astype("float32")/255.0
y_tr_oh = to_categorical(y_tr, 10)
y_te_oh = to_categorical(y_te, 10)

x_tr.shape, y_tr_oh.shape, x_te.shape, y_te_oh.shape

In [ ]:
# PREFILLED: just execute
def build_cnn(clipnorm=None, clipvalue=None, lr=1e-3):
    inputs = layers.Input(shape=(32,32,3))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    opt = tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=clipnorm, clipvalue=clipvalue)
    model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])
    return model

baseline = build_cnn()
hist_base = baseline.fit(x_tr, y_tr_oh, validation_split=0.1, epochs=3, batch_size=128, verbose=2)

In [ ]:
# To-Do: implement a learning rate schedule and train with gradient clipping
# Option 1: use tf.keras.optimizers.schedules.CosineDecay with an initial lr
# model_sched = build_cnn(lr=lr_schedule)
#
# Option 2: step decay via a callback
# def step_decay(epoch, lr):
#     return lr
# cb = tf.keras.callbacks.LearningRateScheduler(step_decay)
#
# Train a clipped model
# model_clip = build_cnn(clipnorm=1.0, lr=1e-3)
# hist_clip = model_clip.fit(...)
#
# Compare histories: plot loss and accuracy for baseline vs clipped or scheduled


**To-Do:** Plot two figures. Figure 1 shows training and validation accuracy per epoch for baseline and your improved run. Figure 2 shows training and validation loss per epoch. Explain in 3 to 5 sentences what changed.


## Exercise 2: Building a Generative Model

**As stated in the exercises**  
Objective. Create a generative model to produce synthetic data.  
Instructions. Choose an architecture such as a GAN or a VAE. Train on a dataset such as MNIST or CelebA. Evaluate sample quality using FID or visual inspection. Experiment with architectures such as DCGAN or loss choices. Write a short reflection on challenges and applications.

**Guidance**  
A VAE on MNIST is simpler to train than a GAN. Implement encoder and decoder with a reparameterization step. Use binary cross entropy reconstruction and KL regularization.


**Learning point**  
VAEs learn a latent distribution that supports interpolation and sampling. GANs can yield sharper images but are sensitive to instability and mode collapse.


In [ ]:
# PREFILLED: just execute
from tensorflow.keras.datasets import mnist
(xm_tr, ym_tr), (xm_te, ym_te) = mnist.load_data()
xm_tr = xm_tr.astype("float32")/255.0
xm_te = xm_te.astype("float32")/255.0
xm_tr = np.expand_dims(xm_tr, -1)
xm_te = np.expand_dims(xm_te, -1)
xm_tr.shape, xm_te.shape

In [ ]:
# To-Do: implement a small VAE
# latent_dim = 16
# Encoder: Input (28,28,1) -> Conv/Dense -> z_mean, z_logvar
# Reparameterize: z = z_mean + exp(0.5*z_logvar) * eps
# Decoder: Dense/ConvTranspose -> output logits or probs
#
# Hints:
#
# Define VAE loss = recon_loss + KL
# ...
# Train for a few epochs and then generate samples from standard normal in latent space


**To-Do:** Display a grid of generated digits and comment on sample diversity and blur. If you choose a GAN instead, show a grid of samples and describe any instability you observed.


![image.png](attachment:image.png)


## Exercise 3: Handling Imbalanced Datasets

**As stated in the exercises**  
Objective. Improve performance on imbalanced datasets using advanced techniques.  
Instructions. Choose an imbalanced dataset such as credit card fraud. Apply SMOTE or ADASYN to balance the data. Train a model with focal loss or weighted cross entropy. Compare precision, recall, and F1 before and after. Write an analysis on the importance of handling imbalance.


**Learning point**  
Accuracy is misleading under class imbalance. Prefer precision, recall, F1, and PR AUC. SMOTE synthesizes minority points and focal loss focuses the objective on hard examples.


In [ ]:
# PREFILLED: just execute
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_classification(n_samples=5000, n_features=20, n_informative=6, n_redundant=2,
                           weights=[0.96, 0.04], flip_y=0.005, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_trs = scaler.fit_transform(X_tr)
X_tes = scaler.transform(X_te)

print("Imbalance:", np.bincount(y_tr), "->", np.bincount(y_te))

In [ ]:
# To-Do: apply SMOTE if available, else implement simple random oversampling of the minority
# try:
#     from imblearn.over_sampling import SMOTE
#     sm = SMOTE(...)
#     X_bal, y_bal = sm.fit_resample(X_trs, y_tr)
#     print("After SMOTE:", np.bincount(y_bal))
# except Exception as e:
#     print("imblearn not available. Using naive oversampling.")
#     idx_min = np.where(y_tr==1)[0]
#     reps = int((len(y_tr)-len(idx_min)) / max(1, len(idx_min)))
#     extra = np.random.choice(idx_min, size=reps*len(idx_min), replace=True)
#     X_bal = np.vstack([X_trs, X_trs[extra]])
#     y_bal = np.concatenate([y_tr, y_tr[extra]])
#     print("After oversampling:", np.bincount(y_bal))

In [ ]:
# To-Do: train a small MLP with either class weights or focal loss
# from tensorflow.keras import layers, models
# def make_mlp():
#     inputs = layers.Input(shape=(X_trs.shape[1],))
#     x = layers.Dense(64, activation="relu")(inputs)
#     x = layers.Dense(32, activation="relu")(x)
#     outputs = layers.Dense(1, activation="sigmoid")(x)
#     m = models.Model(inputs, outputs)
#     m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
#     return m
#
# Option A: class weights
# ....
# print(classification_report(y_te, preds, digits=3))
#
# Option B: focal loss (implement function gamma=2)
# def focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
#     ...
# m = make_mlp()
# m.compile(optimizer="adam", loss=focal_loss, metrics=["accuracy"])
# h = m.fit(X_bal, y_bal, validation_split=0.1, epochs=5, batch_size=256, verbose=2)
# preds = (m.predict(X_tes, verbose=0).ravel() >= 0.5).astype(int)
# print(classification_report(y_te, preds, digits=3))

**To-Do:** Plot a confusion matrix and compute precision, recall, and F1 before and after balancing. Explain in 3 to 5 sentences what changed and why.


## Exercise 4: Model Robustness and Adversarial Training

**As stated in the exercises**  
Objective. Improve robustness against adversarial attacks.  
Instructions. Train a model on MNIST or CIFAR 10. Generate adversarial examples using FGSM or PGD. Apply adversarial training by mixing some adversarial samples into batches. Evaluate robustness on clean and adversarial data. Write a short reflection on the importance and trade offs.


**Learning point**  
Adversarial training improves worst case robustness but can reduce clean accuracy. FGSM uses a single gradient step. PGD uses multiple steps and is stronger.


In [ ]:
# PREFILLED: just execute
from tensorflow.keras.datasets import mnist
(xn_tr, yn_tr), (xn_te, yn_te) = mnist.load_data()
xn_tr = xn_tr.astype("float32")/255.0
xn_te = xn_te.astype("float32")/255.0
xn_tr = np.expand_dims(xn_tr, -1)
xn_te = np.expand_dims(xn_te, -1)

from tensorflow.keras.utils import to_categorical
yn_tr_oh = to_categorical(yn_tr, 10)
yn_te_oh = to_categorical(yn_te, 10)

def make_mnist_cnn():
    inputs = layers.Input(shape=(28,28,1))
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    m = models.Model(inputs, outputs)
    m.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return m

model_clean = make_mnist_cnn()
hist_clean = model_clean.fit(xn_tr, yn_tr_oh, validation_split=0.1, epochs=2, batch_size=128, verbose=2)
clean_acc = float(model_clean.evaluate(xn_te, yn_te_oh, verbose=0)[1])
print({"clean_test_acc": clean_acc})

In [ ]:
# To-Do: implement FGSM and adversarial training
# def fgsm(model, x, y_onehot, eps=0.2):
#     # use tf.GradientTape to get gradient of loss w.r.t inputs
#     # x_adv = clip(x + eps * sign(grad), 0, 1)
#     # return x_adv
#
# x_sample = xn_te[:128]
# y_sample = yn_te_oh[:128]
# x_adv = fgsm(model_clean, x_sample, y_sample, eps=0.2)
# adv_loss, adv_acc = model_clean.evaluate(x_adv, y_sample, verbose=0)
# print({"acc_on_adv_before_adv_training": float(adv_acc)})
#
# Now adversarial training for few epochs by mixing clean and FGSM examples
# model_adv = make_mnist_cnn()
# for epoch in range(2):
#     # create a batch of adversarial examples on the fly from a subset
#     # train on a concatenation of clean and adversarial
#     pass
# adv_clean = float(model_adv.evaluate(xn_te, yn_te_oh, verbose=0)[1])
# x_adv_test = fgsm(model_adv, xn_te[:1000], yn_te_oh[:1000], eps=0.2)
# adv_adv = float(model_adv.evaluate(x_adv_test, yn_te_oh[:1000], verbose=0)[1])
# print({"adv_trained_clean_acc": adv_clean, "adv_trained_adv_acc": adv_adv})

**To-Do:** Show a few clean vs adversarial images side by side and comment on perceptibility. Reflect on the trade off between clean and robust accuracy.


![image.png](attachment:image.png)


## Conclusion

You applied advanced optimization, generative modeling, reinforcement learning, handling of class imbalance, and adversarial robustness. These exercises connect to real production concerns such as stability, data scarcity, safety, and fairness. Extend by adding cosine warm restarts, diffusion models, PPO with generalized advantage estimation, cost sensitive calibration on imbalanced data, and PGD adversarial training.
